# Unit Tests — Data Generator

Validates correctness, domain constraints, referential integrity, and audit field completeness for all six generated entities.

| Entity | Tests |
|--------|-------|
| Customers | ID prefix, email format, city/tier/payment validity, coordinates, audit fields |
| Restaurants | ID uniqueness, cuisine list, rating precision, boolean flags |
| Menu Items | Price range, veg/non-veg options, prep time, restaurant linkage |
| Orders | Status values, payment integrity, subtotal vs line items, referential integrity |
| Deliveries | Time range, vehicle type, status, linkage to deliverable orders only |
| Reviews | Rating range, sentiment derivation, links to delivered orders only |

In [ ]:
from datetime import datetime
from data_generator.config import DataVolumeConfig, ZomatoDataConfig
from data_generator.generators import (
    generate_customers, generate_restaurants, generate_menu_items,
    generate_orders, generate_deliveries, generate_reviews,
)

volume = DataVolumeConfig(
    num_customers=100, num_restaurants=20, num_orders=500,
    num_deliveries=400, num_reviews=300, num_menu_items_per_restaurant=15,
)
cfg = ZomatoDataConfig()

customers   = generate_customers(volume, cfg)
restaurants = generate_restaurants(volume, cfg)
menu_items  = generate_menu_items(restaurants, volume, cfg)
orders, order_items = generate_orders(customers, restaurants, menu_items, volume, cfg)
deliveries  = generate_deliveries(orders, cfg)
reviews     = generate_reviews(orders, volume, cfg)

PASS = "\u2713 PASS"
FAIL = "\u2717 FAIL"

results = []

def check(name, condition):
    status = PASS if condition else FAIL
    results.append((name, status))
    print(f"{status}  {name}")

print("Data generated successfully.")
print(f"  customers={len(customers)}, restaurants={len(restaurants)}, menu_items={len(menu_items)}")
print(f"  orders={len(orders)}, order_items={len(order_items)}, deliveries={len(deliveries)}, reviews={len(reviews)}")

## Customer Tests

In [ ]:
check("Customers — correct count",           len(customers) == volume.num_customers)
check("Customers — IDs are unique",          len({c['customer_id'] for c in customers}) == len(customers))
check("Customers — ID prefix CUST_",         all(c['customer_id'].startswith('CUST_') for c in customers[:20]))
check("Customers — email is lowercase",      all(c['email'] == c['email'].lower() for c in customers[:20]))
check("Customers — email contains @",        all('@' in c['email'] for c in customers[:20]))
check("Customers — subscription tier valid", all(c['subscription_tier'] in set(cfg.subscription_tiers) for c in customers))
check("Customers — city in known list",      all(c['city'] in set(cfg.cities) for c in customers))
check("Customers — payment method valid",    all(c['preferred_payment'] in set(cfg.payment_methods) for c in customers))
check("Customers — cuisine valid",           all(c['preferred_cuisine'] in set(cfg.cuisines) for c in customers))
check("Customers — avg_order_value >= 0",   all(c['avg_order_value'] >= 0 for c in customers))
check("Customers — orders_lifetime >= 0",   all(c['total_orders_lifetime'] >= 0 for c in customers))
check("Customers — is_active is bool",       all(isinstance(c['is_active'], bool) for c in customers[:20]))
check("Customers — source system correct",  all(c['_source_system'] == 'zomato_app_backend' for c in customers[:10]))
check("Customers — signup_date valid ISO",   all(datetime.fromisoformat(c['signup_date']) is not None for c in customers[:10]))
check("Customers — coordinates valid range",all(-90 <= c['latitude'] <= 90 and -180 <= c['longitude'] <= 180 for c in customers[:20]))

## Restaurant Tests

In [ ]:
check("Restaurants — correct count",          len(restaurants) == volume.num_restaurants)
check("Restaurants — IDs are unique",         len({r['restaurant_id'] for r in restaurants}) == len(restaurants))
check("Restaurants — ID prefix REST_",        all(r['restaurant_id'].startswith('REST_') for r in restaurants[:10]))
check("Restaurants — cuisines is list",       all(isinstance(r['cuisines'], list) for r in restaurants))
check("Restaurants — at least 1 cuisine",     all(len(r['cuisines']) >= 1 for r in restaurants))
check("Restaurants — cuisines valid",         all(c in set(cfg.cuisines) for r in restaurants for c in r['cuisines']))
check("Restaurants — rating 2.5-5.0",         all(2.5 <= r['rating'] <= 5.0 for r in restaurants))
check("Restaurants — rating 1-decimal",       all(round(r['rating'], 1) == r['rating'] for r in restaurants))
check("Restaurants — type valid",             all(r['restaurant_type'] in set(cfg.restaurant_types) for r in restaurants))
check("Restaurants — city in known list",     all(r['city'] in set(cfg.cities) for r in restaurants))
check("Restaurants — total_reviews >= 10",    all(r['total_reviews'] >= 10 for r in restaurants))
check("Restaurants — boolean flags correct",  all(isinstance(r['is_delivering_now'], bool) for r in restaurants[:10]))
check("Restaurants — source system correct",  all(r['_source_system'] == 'restaurant_onboarding_svc' for r in restaurants[:10]))

## Menu Item Tests

In [ ]:
restaurant_ids = {r['restaurant_id'] for r in restaurants}
menu_restaurant_ids = {i['restaurant_id'] for i in menu_items}

check("Menu Items — covers all restaurants", menu_restaurant_ids == restaurant_ids)
check("Menu Items — prices in range",        all(cfg.min_item_price <= i['price'] <= cfg.max_item_price for i in menu_items[:50]))
check("Menu Items — item_id not null",       all(i['item_id'] is not None and i['item_id'] != '' for i in menu_items[:50]))
check("Menu Items — veg_nonveg valid",       all(i['veg_nonveg'] in {'Veg', 'Non-Veg', 'Egg'} for i in menu_items[:50]))
check("Menu Items — is_available bool",      all(isinstance(i['is_available'], bool) for i in menu_items[:50]))
check("Menu Items — is_bestseller bool",     all(isinstance(i['is_bestseller'], bool) for i in menu_items[:50]))
check("Menu Items — prep time valid",        all(i['preparation_time_mins'] in {10,15,20,25,30,40,45} for i in menu_items[:50]))

## Order Tests

In [ ]:
customer_ids    = {c['customer_id'] for c in customers}
restaurant_ids  = {r['restaurant_id'] for r in restaurants}
order_ids       = {o['order_id'] for o in orders}

order_line_totals: dict = {}
for item in order_items:
    order_line_totals.setdefault(item['order_id'], 0.0)
    order_line_totals[item['order_id']] += item['line_total']

subtotal_match = all(
    abs(o['subtotal'] - round(order_line_totals.get(o['order_id'], o['subtotal']), 2)) < 0.02
    for o in orders[:5] if o['order_id'] in order_line_totals
)

check("Orders — generated with items",            len(orders) > 0 and len(order_items) > 0)
check("Orders — IDs are unique",                  len(order_ids) == len(orders))
check("Orders — ID prefix ORD_",                  all(o['order_id'].startswith('ORD_') for o in orders[:10]))
check("Orders — total_amount >= 0",               all(o['total_amount'] >= 0 for o in orders[:50]))
check("Orders — status valid",                    all(o['order_status'] in set(cfg.order_statuses) for o in orders))
check("Orders — payment method valid",            all(o['payment_method'] in set(cfg.payment_methods) for o in orders[:50]))
check("Orders — platform valid",                  all(o['platform'] in set(cfg.order_platforms) for o in orders[:50]))
check("Orders — customer FK valid",               all(o['customer_id'] in customer_ids for o in orders[:50]))
check("Orders — restaurant FK valid",             all(o['restaurant_id'] in restaurant_ids for o in orders[:50]))
check("Orders — items link to orders",            all(i['order_id'] in order_ids for i in order_items[:50]))
check("Orders — subtotal matches line items",     subtotal_match)
check("Orders — discount_amount >= 0",            all(o['discount_amount'] >= 0 for o in orders[:50]))
check("Orders — tax_amount >= 0",                 all(o['tax_amount'] >= 0 for o in orders[:50]))
check("Orders — delivered → payment Completed",  all(o['payment_status'] == 'Completed' for o in orders if o['order_status'] == 'Delivered'))
check("Orders — source system correct",           all(o['_source_system'] == 'order_management_svc' for o in orders[:10]))

## Delivery Tests

In [ ]:
deliverable_statuses  = {'Out for Delivery', 'Delivered'}
delivery_order_ids    = {d['order_id'] for d in deliveries}

check("Deliveries — generated > 0",             len(deliveries) > 0)
check("Deliveries — IDs unique",                len({d['delivery_id'] for d in deliveries}) == len(deliveries))
check("Deliveries — ID prefix DLV_",            all(d['delivery_id'].startswith('DLV_') for d in deliveries[:10]))
check("Deliveries — time > 0",                  all(d['delivery_time_mins'] > 0 for d in deliveries[:50]))
check("Deliveries — time in range 15-60",       all(15 <= d['delivery_time_mins'] <= 60 for d in deliveries[:50]))
check("Deliveries — vehicle type valid",        all(d['vehicle_type'] in {'Bike','Scooter','Bicycle','Car'} for d in deliveries[:50]))
check("Deliveries — status valid",              all(d['delivery_status'] in set(cfg.delivery_statuses) for d in deliveries[:50]))
check("Deliveries — order FK valid",            all(d['order_id'] in order_ids for d in deliveries[:50]))
check("Deliveries — only deliverable orders",   all(o['order_status'] in deliverable_statuses for o in orders if o['order_id'] in delivery_order_ids))
check("Deliveries — distance > 0",              all(d['delivery_distance_km'] > 0 for d in deliveries[:50]))
check("Deliveries — rating valid or null",      all(d['delivery_rating'] is None or 1 <= d['delivery_rating'] <= 5 for d in deliveries[:50]))
check("Deliveries — source system correct",     all(d['_source_system'] == 'logistics_svc' for d in deliveries[:10]))

## Review Tests

In [ ]:
delivered_order_ids = {o['order_id'] for o in orders if o['order_status'] == 'Delivered'}

def sentiment_ok(r):
    if r['rating'] >= 4: return r['sentiment_label'] == 'Positive'
    if r['rating'] <= 2: return r['sentiment_label'] == 'Negative'
    return r['sentiment_label'] == 'Neutral'

check("Reviews — generated > 0",             len(reviews) > 0)
check("Reviews — ID prefix REV_",            all(r['review_id'].startswith('REV_') for r in reviews[:10]))
check("Reviews — ratings 1-5",               all(1 <= r['rating'] <= 5 for r in reviews[:50]))
check("Reviews — ratings are integers",      all(isinstance(r['rating'], int) for r in reviews[:50]))
check("Reviews — sentiment label correct",   all(sentiment_ok(r) for r in reviews[:50]))
check("Reviews — is_verified always True",   all(r['is_verified_order'] is True for r in reviews))
check("Reviews — only delivered orders",     all(r['order_id'] in delivered_order_ids for r in reviews))
check("Reviews — upvotes >= 0",              all(r['upvotes'] >= 0 for r in reviews[:50]))
check("Reviews — has_photo is bool",         all(isinstance(r['has_photo'], bool) for r in reviews[:50]))
check("Reviews — review_text not empty",     all(r['review_text'] and len(r['review_text'].strip()) > 0 for r in reviews[:50]))
check("Reviews — source system correct",     all(r['_source_system'] == 'review_svc' for r in reviews[:10]))

## Summary

In [ ]:
passed = sum(1 for _, s in results if s == PASS)
failed = sum(1 for _, s in results if s == FAIL)
total  = len(results)

print(f"\n{'='*55}")
print(f"  Data Generator Test Results")
print(f"{'='*55}")
print(f"  Total  : {total}")
print(f"  Passed : {passed}")
print(f"  Failed : {failed}")
print(f"  Result : {'ALL TESTS PASSED' if failed == 0 else f'{failed} TEST(S) FAILED'}")
print(f"{'='*55}")

if failed > 0:
    print("\nFailed tests:")
    for name, status in results:
        if status == FAIL:
            print(f"  {status}  {name}")
    raise AssertionError(f"{failed} test(s) failed — see details above.")